# Calibrating the JAX 2D-NS generator against the MNO download

The Zenodo MNO generator (`2D_NS_Re*.npy`) is **not published**, so we calibrate
`generate_fno_ns_jax.py` to reproduce its *statistics* rather than its exact code.
Reference = the downloaded `2D_NS_Re500.npy`. We match three things:
1. **per-frame std** (steady value + transient from the IC),
2. **vorticity spectrum** shape,
3. **frame-to-frame autocorrelation** (sets `record-dt`).

**Result:** MNO flow = `-4cos(4y)` (k=4, amplitude 4) forcing with a *small* drag ~0.01
(no drag blows up; kf_2d's 0.1 over-damps to a k=4 peak); `nu=1/Re`. Steady std ~4.0-4.5
(download 4.78), spectrum peak k=1 (condensate), and `record-dt ~= 0.34` t.u. matches rho1~0.61.

In [ ]:
import numpy as np, jax, jax.numpy as jnp, os
import matplotlib.pyplot as plt
from data_generation.generate_fno_ns_jax import (
    gaussian_random_field, build_operators, forcing_field, make_integrator)

REF = 'flow-data/2D_NS_Re500.npy'   # downloaded MNO array (mmap a subset; works on a partial file)
NSAMP = 40

def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

with open(REF, 'rb') as f:
    v = np.lib.format.read_magic(f)
    shape, _, dt = np.lib.format._read_array_header(f, v); off = f.tell()
    nT, T, H, W = shape; bpf = H * W * dt.itemsize
    nav = (os.path.getsize(REF) - off) // (T * bpf)
    ns = min(NSAMP, int(nav)); f.seek(off)
    ref = np.frombuffer(f.read(ns * T * bpf), dtype=dt).reshape(ns, T, H, W)
std_curve = ref.std(axis=(0, 2, 3))
rho1 = np.mean([corr(ref[i, t], ref[i, t + 1]) for i in range(ns) for t in range(300, 499, 5)])
print(f'download: {ns} traj | frame0 std={std_curve[0]:.3f}  steady std={std_curve[300:].mean():.3f}  rho1={rho1:.3f}')

In [ ]:
# reference diagnostics: std-vs-frame transient + mean vorticity spectrum
kk = np.fft.fftfreq(H, 1 / H)
Kref = np.round(np.sqrt(np.add.outer(kk ** 2, kk ** 2))).astype(int)
Ek_ref = np.zeros(H)
for i in range(ns):
    for t in range(300, T, 20):
        Ek_ref += np.bincount(Kref.ravel(), np.abs(np.fft.fft2(ref[i, t])).ravel() ** 2, minlength=H)
Ek_ref = Ek_ref[1:H // 2]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(std_curve); ax[0].axhline(std_curve[300:].mean(), ls='--', c='k')
ax[0].set(xlabel='frame', ylabel='std', title='download std vs frame (transient from IC)')
ax[1].loglog(np.arange(1, H // 2), Ek_ref)
ax[1].set(xlabel='k', ylabel='E(k)', title='download vorticity spectrum'); ax[1].grid(which='both', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# generator trace helper (Re500, float32)
n = 64; visc = 1 / 500.0; DT = 5e-4; dtype = jnp.float32
kx, ky, ksq, ksqnz, deal, xx, yy = build_operators(n, dtype)

def trace(drag, spinup_t=15.0, rec_every=400, rec_steps=80, seed=2):
    fH = jnp.fft.fft2(-4.0 * jnp.cos(4 * yy))
    integ = make_integrator(n, visc, DT, drag, fH, deal, kx, ky, ksq, ksqnz,
                            int(spinup_t / DT), rec_steps, rec_every)
    w0 = gaussian_random_field(jax.random.PRNGKey(seed), 4, n).astype(dtype)
    return np.asarray(integ(w0))   # (rec_steps, 4, n, n)

In [ ]:
# drag sweep: which drag matches the download's steady std + k=1 spectrum peak
for d in [0.1, 0.05, 0.03, 0.02, 0.01]:
    fr = trace(d, spinup_t=0.0, rec_every=400, rec_steps=80)
    Ek = np.bincount(Kref.ravel(), np.abs(np.fft.fft2(fr[-1, 0])).ravel() ** 2)[1:]
    print(f'drag={d:<5}: steady std={fr[-5:].std():.3f}  spectrum peak k={1 + int(np.argmax(Ek))}')
print(f'(download target: std={std_curve[300:].mean():.2f}, peak k=1)')

In [ ]:
# record-dt: physical time lag where generator autocorr == download rho1
fr = trace(0.01, spinup_t=15.0, rec_every=40, rec_steps=300)
fine_dt = 40 * DT
lags = list(range(1, 150))
rho = np.array([np.mean([corr(fr[t, i], fr[t + L, i]) for i in range(4) for t in range(0, 300 - L, 9)]) for L in lags])
times = np.array(lags) * fine_dt
rec_dt = float(times[int(np.argmin(np.abs(rho - rho1)))])
plt.plot(times, rho); plt.axhline(rho1, ls='--', c='r', label=f'download rho1={rho1:.2f}')
plt.axvline(rec_dt, ls=':', c='k', label=f'record-dt*={rec_dt:.3f}')
plt.xlabel('time lag (t.u.)'); plt.ylabel('autocorr'); plt.legend(); plt.title('frame-spacing calibration'); plt.show()
print(f'CALIBRATED: forcing=mno (k=4, drag=0.01), nu=1/Re, dt={DT}, record-dt~={rec_dt:.3f} (record-every~={int(round(rec_dt/DT))} steps)')

## Launch (calibrated)
```bash
python data_generation/generate_fno_ns_jax.py --forcing mno --re 500 --res 64 \
    --n-samples <N> --record-steps 501 --dt 5e-4 --record-dt 0.34 --dtype float32 \
    --out 2D_NS_Re500_regen.npy
```
**Caveats:** steady std ~4.0-4.5 vs download 4.78 (~5-10% low); IC std differs slightly. Good for
OOD stress-testing, not bit-exact. **Cost:** record-every ~680 steps/frame x 501 frames is heavy
(full 1000-traj MNO scale = many hours/days); a few dozen trajectories is plenty for OOD. Re40/Re5000
need their own calibration (different steady std; Re5000 at 128 needs a smaller dt for CFL).